# 04 - Model Training

This notebook trains multiple machine learning models for customer churn prediction and compares their performance.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 2. Load Processed Data

In [ ]:
# Load training and testing data
train_data = pd.read_csv('../data/processed/train_data.csv')
test_data = pd.read_csv('../data/processed/test_data.csv')

# Separate features and target
X_train = train_data.drop('Churn', axis=1)
y_train = train_data['Churn']
X_test = test_data.drop('Churn', axis=1)
y_test = test_data['Churn']

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")
print(f"\nTraining churn rate: {y_train.mean() * 100:.2f}%")
print(f"Testing churn rate: {y_test.mean() * 100:.2f}%")

## 3. Define Models

In [ ]:
# Define models to train
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100),
    'XGBoost': XGBClassifier(random_state=42, n_estimators=100, eval_metric='logloss'),
    'Support Vector Machine': SVC(random_state=42, probability=True)
}

print(f"Models to train: {list(models.keys())}")

## 4. Train Models and Evaluate

In [ ]:
# Dictionary to store results
results = {}
trained_models = {}

# Train and evaluate each model
for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Training {name}...")
    print('='*60)
    
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
    
    # Store results
    results[name] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': roc_auc
    }
    
    # Store trained model
    trained_models[name] = model
    
    # Print results
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    if roc_auc:
        print(f"ROC-AUC: {roc_auc:.4f}")

## 5. Compare Model Performance

In [ ]:
# Create a DataFrame to compare results
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('roc_auc', ascending=False)

print("\nModel Performance Comparison:")
print(results_df)

## 6. Select Best Model

In [ ]:
# Select best model based on ROC-AUC score
best_model_name = results_df['roc_auc'].idxmax()
best_model = trained_models[best_model_name]

print(f"\n{'='*60}")
print(f"BEST MODEL: {best_model_name}")
print('='*60)
print(f"ROC-AUC Score: {results_df.loc[best_model_name, 'roc_auc']:.4f}")
print(f"Accuracy: {results_df.loc[best_model_name, 'accuracy']:.4f}")
print(f"Recall: {results_df.loc[best_model_name, 'recall']:.4f}")
print(f"F1-Score: {results_df.loc[best_model_name, 'f1_score']:.4f}")

## 7. Detailed Evaluation of Best Model

In [ ]:
# Make predictions with best model
y_pred_best = best_model.predict(X_test)
y_pred_proba_best = best_model.predict_proba(X_test)[:, 1]

# Confusion Matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred_best)
print(cm)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))

## 8. Feature Importance (for tree-based models)

In [ ]:
# Check if best model has feature importance
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': best_model.feature_importances_
    })
    feature_importance = feature_importance.sort_values('importance', ascending=False)
    
    print("\nTop 10 Important Features:")
    print(feature_importance.head(10))
else:
    print("\nFeature importance not available for this model type.")

## 9. Save Best Model

In [ ]:
# Create models directory if it doesn't exist
os.makedirs('../models/', exist_ok=True)

# Save the best model
with open('../models/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Save model results
with open('../models/model_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print(f"\nBest model saved as: ../models/best_model.pkl")
print(f"Model results saved as: ../models/model_results.pkl")

## 10. Training Summary

In [ ]:
print("="*60)
print("MODEL TRAINING SUMMARY")
print("="*60)
print(f"\nTotal Models Trained: {len(models)}")
print(f"Best Model: {best_model_name}")
print(f"Best ROC-AUC Score: {results_df.loc[best_model_name, 'roc_auc']:.4f}")
print(f"Best Recall Score: {results_df.loc[best_model_name, 'recall']:.4f}")

print("\nAll Model Performance:")
for model_name in results_df.index:
    print(f"\n{model_name}:")
    print(f"  Accuracy: {results_df.loc[model_name, 'accuracy']:.4f}")
    print(f"  Precision: {results_df.loc[model_name, 'precision']:.4f}")
    print(f"  Recall: {results_df.loc[model_name, 'recall']:.4f}")
    print(f"  F1-Score: {results_df.loc[model_name, 'f1_score']:.4f}")
    print(f"  ROC-AUC: {results_df.loc[model_name, 'roc_auc']:.4f}")

print("="*60)

## Summary

This notebook completed model training:
- Trained 6 different machine learning models
- Evaluated each model using multiple metrics (Accuracy, Precision, Recall, F1-Score, ROC-AUC)
- Compared model performance
- Selected the best model based on ROC-AUC score
- Analyzed feature importance (for applicable models)
- Saved the best model and results for deployment

Models trained:
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Gradient Boosting
5. XGBoost
6. Support Vector Machine

The best model has been saved and is ready for detailed evaluation and deployment.